In [15]:
import pandas as pd
properties_data = pd.read_csv('../data_to_use/data/preprocessed_data/cleaned_properties.csv')
# rename 'property_id' column to 'id' to match amenities_data
properties_data.rename(columns={'property_id': 'id'}, inplace=True)
properties_data.head()

,id,latitude,longitude,street_address,suburb,postcode,weekly_rent_number,property_type,bedrooms,bathrooms,...,services_available_pay_tv,services_available_solar_power,services_available_water,services_heating,spa_outside,spa_spa,sport_gym,sport_pool,sport_tennis_court,views_garden_views
0,515812,-37.869108,144.990422,"108/126 Chapel Street,",st kilda,3182,550.0,Apartment,1,1,...,0,0,0,0,0,0,0,0,0,0
1,802350,-37.802986,144.961405,"24 Barkly Place,",carlton,3053,360.0,Studio,1,1,...,0,0,0,0,0,0,0,0,0,0
2,802356,-37.808700,144.984130,"45 Victoria Parade,",collingwood,3066,360.0,Studio,1,1,...,0,0,0,0,0,0,0,0,0,0
3,802372,-37.714742,145.060900,"116-130 Main Drive,",macleod,3085,250.0,Studio,0,1,...,0,0,0,0,0,0,0,0,0,0
4,802379,-37.875928,145.046426,"117/60 Waverley Road,",malvern east,3145,540.0,Apartment,2,1,...,0,0,0,0,0,0,0,0,0,0


In [16]:
sa2_panel = pd.read_csv('../data_to_use/data/sa2_panel/sa2_panel.csv')
sa2_panel.head()

,sa2_code_2021,sa2_name,year,feature,value,status
0,201011481,Ballarat East - Warrenheip,2001,population,8018.0,observed
1,201011481,Ballarat East - Warrenheip,2002,population,8066.0,observed
2,201011481,Ballarat East - Warrenheip,2003,population,8080.0,observed
3,201011481,Ballarat East - Warrenheip,2004,population,8061.0,observed
4,201011481,Ballarat East - Warrenheip,2005,population,8057.0,observed


In [27]:
population_data = sa2_panel.loc[sa2_panel['feature'] == 'population']
population_data.drop(columns=['year'], inplace=True)
population_data.head(1)

,sa2_code_2021,sa2_name,feature,value,status
0,201011481,Ballarat East - Warrenheip,population,8018.0,observed


# Additional data
- Household dwelling data: `data/forecasting/Household_Dwelling_Forecast_2136_CleanedV2.csv`

In [31]:
dwellings_data = pd.read_csv('../data_to_use/data/forecasting/Household_Dwelling_Forecast_2136_CleanedV2.csv')
cols_to_keep = ['SA2_Code', 'Dwellings_2021', 'Dwellings_2026', 'Dwellings_2031', 'Dwellings_2036']
dwellings_data = dwellings_data[cols_to_keep]
dwellings_data.head()

,SA2_Code,Dwellings_2021,Dwellings_2026,Dwellings_2031,Dwellings_2036
0,201011001,6245.0,8252.000000,9732.000000,10830.000000
1,201011002,5970.0,6134.548371,6350.037451,6553.095884
2,201011005,2768.0,2943.325691,3199.637967,3445.433425
3,201011006,4172.0,6585.376102,8740.266903,10770.851234
4,201011007,1490.0,1600.000000,1720.000000,1856.363636


In [33]:
import geopandas as gpd
from shapely.geometry import Point
def find_SA2(data, latitude, longitude):
    """
    This function takes in latitude and longitude coordinates 
    and outputs the Statistical Area Level 2 (SA2) that contains 
    the point.

    Args:
        data: The geospatial data containing the SA2 polygons
        latitude: Latitude coordinate as a decimal
        longitude: Longitidue coordinate as a decimal

    Returns:
        String containing name of SA2 area according to ABS.
    """
    # Create point from longitude and latitude
    point = gpd.GeoDataFrame(
        geometry=[Point(longitude, latitude)],
        crs="EPSG:4326"
    )

    # Make sure the point uses the same CRS as the SA2 polygons
    point = point.to_crs(data.crs)

    # Find the SA2 containing the point
    match = gpd.sjoin(
        point,
        data[["SA2_NAME21", "geometry"]],
        how="left",
        predicate="within"
    )

    # Return SA2 name
    if match["SA2_NAME21"].notna().any():
        return match["SA2_NAME21"].iloc[0]
    else:
        return None

In [34]:
geospatial_sa2_data = gpd.read_file('../data_to_use/data/geospatial_data/Vic_Area_Zones/Vic_Area_Zones.shp')
latitudes = properties_data['latitude'].tolist()
longitudes = properties_data['longitude'].tolist()
sa2_names = [find_SA2(geospatial_sa2_data, lat, lon) for lat, lon in zip(latitudes, longitudes)]


In [35]:
print(sa2_names)

['St Kilda East', 'Carlton', 'Collingwood', 'Kingsbury', 'Malvern East', 'Prahran - Windsor', 'Kingsbury', 'Kingsbury', 'Melbourne CBD - East', 'Malvern East', 'Kingsbury', 'Kingsbury', 'Carlton', 'Footscray', 'Footscray', 'Carlton', 'Carlton', 'Noble Park - West', 'Malvern East', 'Prahran - Windsor', 'Clayton (North) - Notting Hill', 'Prahran - Windsor', 'Kingsbury', 'North Melbourne', 'Carlton', 'Malvern East', 'Carlton', 'North Melbourne', 'Carlton', 'Kingsbury', 'Kingsbury', 'Kingsbury', 'Kingsbury', 'Kingsbury', 'Burwood (Vic.)', 'Kingsbury', 'Kingsbury', 'Brighton (Vic.)', 'Footscray', 'Carlton', 'Ivanhoe', 'Footscray', 'Clayton South', 'Clayton - Central', 'Pascoe Vale', 'Elsternwick', 'Carlton', 'Clayton (North) - Notting Hill', 'Pascoe Vale South', 'Bundoora - West', 'Altona', 'Malvern East', 'Brunswick East', 'Brunswick - North', 'Malvern East', 'Yarriambiack', 'Essendon - East', 'Oakleigh - Huntingdale', 'Gladstone Park - Westmeadows', 'Brunswick - South', 'Coburg North', 'P

In [18]:
amenities_data = pd.read_csv('../data_to_use/data/amenity_data/listings_with_amenity_distances.csv')
amenities_data.head()

,id,url,latitude,longitude,suburb,nearest_school_km,school_count_2km,nearest_train_station_km,nearest_park_km,park_count_2km,nearest_supermarket_km,supermarket_count_4km,nearest_tram_stop_km,tram_stop_capped,nearest_bus_stop_km,bus_stop_capped
0,188706,https://www.rent.com.au/property/1007-20-rakai...,-37.814688,144.941083,docklands,0.2239,4,1.7541,0.3272,27,0.2927,47,0.2090,False,0.4570,False
1,194172,https://www.rent.com.au/property/4408-601-litt...,-37.813699,144.953926,melbourne,0.2451,7,0.5049,0.1751,42,0.5418,50,0.1963,False,0.1793,False
2,430370,https://www.rent.com.au/property/29-bond-st-cl...,-37.930804,145.112401,clayton south,0.8481,4,1.5133,0.4956,32,0.8235,13,3.0000,True,0.2838,False
3,515812,https://www.rent.com.au/property/108-126-chape...,-37.869108,144.990422,st kilda,0.6413,11,0.2585,0.4311,35,0.2379,24,0.0904,False,0.3271,False
4,519229,https://www.rent.com.au/property/33-2-greenfie...,-37.923180,145.128629,clayton,0.8972,3,1.1356,0.7598,17,1.0678,12,3.0000,True,0.7769,False


In [19]:
raw_properties_data = pd.read_csv('../data_to_use/data/properties_db/properties.csv')

In [20]:
print(len(raw_properties_data))
missing_ids = set(amenities_data['id']) - set(raw_properties_data['id'])
print(len(missing_ids))

9301
0


In [21]:
# left join properties_data with amenities_data 
merged_data = pd.merge(properties_data, amenities_data, on='id', how='left')
merged_data.rename(columns={'latitude_x': 'latitude', 'longitude_x': 'longitude', 'suburb_x': 'suburb'}, inplace=True)
print(merged_data.shape)
merged_data.head()

(9145, 157)


,id,latitude,longitude,street_address,suburb,postcode,weekly_rent_number,property_type,bedrooms,bathrooms,...,school_count_2km,nearest_train_station_km,nearest_park_km,park_count_2km,nearest_supermarket_km,supermarket_count_4km,nearest_tram_stop_km,tram_stop_capped,nearest_bus_stop_km,bus_stop_capped
0,515812,-37.869108,144.990422,"108/126 Chapel Street,",st kilda,3182,550.0,Apartment,1,1,...,11.0,0.2585,0.4311,35.0,0.2379,24.0,0.0904,False,0.3271,False
1,802350,-37.802986,144.961405,"24 Barkly Place,",carlton,3053,360.0,Studio,1,1,...,10.0,0.5398,0.2852,50.0,0.4218,50.0,0.3539,False,0.4015,False
2,802356,-37.808700,144.984130,"45 Victoria Parade,",collingwood,3066,360.0,Studio,1,1,...,9.0,0.9842,0.4145,42.0,0.6940,50.0,0.1655,False,0.0798,False
3,802372,-37.714742,145.060900,"116-130 Main Drive,",macleod,3085,250.0,Studio,0,1,...,0.0,3.1039,0.4513,23.0,1.4477,10.0,1.3952,False,0.0535,False
4,802379,-37.875928,145.046426,"117/60 Waverley Road,",malvern east,3145,540.0,Apartment,2,1,...,7.0,1.0023,0.8585,11.0,1.0210,25.0,0.1384,False,0.1764,False


In [22]:
X = merged_data.copy()
cols_to_drop = ['street_address', 'url', 'suburb_y']
X.drop(columns=cols_to_drop, inplace=True)
y = X.pop('weekly_rent_number')
X.head()

,id,latitude,longitude,suburb,postcode,property_type,bedrooms,bathrooms,ensuites,car_spaces,...,school_count_2km,nearest_train_station_km,nearest_park_km,park_count_2km,nearest_supermarket_km,supermarket_count_4km,nearest_tram_stop_km,tram_stop_capped,nearest_bus_stop_km,bus_stop_capped
0,515812,-37.869108,144.990422,st kilda,3182,Apartment,1,1,0.0,1,...,11.0,0.2585,0.4311,35.0,0.2379,24.0,0.0904,False,0.3271,False
1,802350,-37.802986,144.961405,carlton,3053,Studio,1,1,0.0,0,...,10.0,0.5398,0.2852,50.0,0.4218,50.0,0.3539,False,0.4015,False
2,802356,-37.808700,144.984130,collingwood,3066,Studio,1,1,0.0,0,...,9.0,0.9842,0.4145,42.0,0.6940,50.0,0.1655,False,0.0798,False
3,802372,-37.714742,145.060900,macleod,3085,Studio,0,1,0.0,0,...,0.0,3.1039,0.4513,23.0,1.4477,10.0,1.3952,False,0.0535,False
4,802379,-37.875928,145.046426,malvern east,3145,Apartment,2,1,0.0,0,...,7.0,1.0023,0.8585,11.0,1.0210,25.0,0.1384,False,0.1764,False


In [23]:
from sklearn.preprocessing import OneHotEncoder

# One-hot encode suburb
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
suburb_encoded = encoder.fit_transform(X[['suburb']])

# Convert encoded data to DataFrame
suburb_encoded_df = pd.DataFrame(
    suburb_encoded,
    columns=encoder.get_feature_names_out(['suburb']),
    index=X.index
)

# Add encoded suburb columns to X
X = pd.concat([X.drop(columns=['suburb']), suburb_encoded_df], axis=1)

X.head()

,id,latitude,longitude,postcode,property_type,bedrooms,bathrooms,ensuites,car_spaces,furnished,...,suburb_woodend north,suburb_woodstock,suburb_woodvale,suburb_wy yung,suburb_wycheproof,suburb_wyndham vale,suburb_yallambie,suburb_yarraville,suburb_yarrawonga,suburb_yinnar
0,515812,-37.869108,144.990422,3182,Apartment,1,1,0.0,1,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,802350,-37.802986,144.961405,3053,Studio,1,1,0.0,0,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,802356,-37.808700,144.984130,3066,Studio,1,1,0.0,0,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,802372,-37.714742,145.060900,3085,Studio,0,1,0.0,0,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,802379,-37.875928,145.046426,3145,Apartment,2,1,0.0,0,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
# one hot encode property_type
property_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
property_encoded = property_encoder.fit_transform(X[['property_type']])

# Convert encoded data to DataFrame
property_encoded_df = pd.DataFrame(
    property_encoded,
    columns=property_encoder.get_feature_names_out(['property_type']),
    index=X.index
)

# Add encoded property_type columns to X
X = pd.concat(
    [X.drop(columns=['property_type']), property_encoded_df],
    axis=1
)

# Check the result
X.head()

,id,latitude,longitude,postcode,bedrooms,bathrooms,ensuites,car_spaces,furnished,partially_furnished,...,property_type_Flat,property_type_House,property_type_Other,property_type_Serviced Apartment,property_type_Studio,property_type_Terrace,property_type_Townhouse,property_type_Unit,property_type_Villa,property_type_Warehouse
0,515812,-37.869108,144.990422,3182,1,1,0.0,1,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,802350,-37.802986,144.961405,3053,1,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,802356,-37.808700,144.984130,3066,1,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,802372,-37.714742,145.060900,3085,0,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,802379,-37.875928,145.046426,3145,2,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
string_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()
from sklearn.preprocessing import LabelEncoder

for col in string_columns:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col].astype(str))

X.head()

,id,latitude,longitude,postcode,bedrooms,bathrooms,ensuites,car_spaces,furnished,partially_furnished,...,property_type_Flat,property_type_House,property_type_Other,property_type_Serviced Apartment,property_type_Studio,property_type_Terrace,property_type_Townhouse,property_type_Unit,property_type_Villa,property_type_Warehouse
0,515812,-37.869108,144.990422,3182,1,1,0.0,1,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,802350,-37.802986,144.961405,3053,1,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,802356,-37.808700,144.984130,3066,1,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,802372,-37.714742,145.060900,3085,0,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,802379,-37.875928,145.046426,3145,2,1,0.0,0,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Split X and y into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create the Random Forest Regressor
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Display results
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.2f}")

MAE:  88.44
RMSE: 225.99
R²:   0.45
